# 02 — Content Intelligence SQL

**Engine:** DuckDB (in-memory)  
**Scope:** 10 business SQL queries on live TMDB-schema data


In [ ]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import json

# Load data
trending = pd.read_csv("../data/trending_movies_latest.csv")
tv = pd.read_csv("../data/popular_tv_latest.csv")
genres = pd.read_csv("../data/movie_genres_latest.csv")
genre_pop = pd.read_csv("../data/genre_popularity_latest.csv")
upcoming = pd.read_csv("../data/upcoming_movies_latest.csv")
top_rated = pd.read_csv("../data/top_rated_movies_latest.csv")

def parse_genres(s):
    try: return json.loads(s) if isinstance(s, str) else []
    except: return []

trending["genre_ids_list"] = trending["genre_ids"].apply(parse_genres)
tv["genre_ids_list"] = tv["genre_ids"].apply(parse_genres)

con = duckdb.connect(":memory:")
con.register("trending_movies", trending)
con.register("popular_tv", tv)
con.register("movie_genres", genres)
con.register("genre_popularity", genre_pop)
con.register("upcoming_movies", upcoming)
con.register("top_rated_movies", top_rated)

con.execute("CREATE TABLE genre_lookup AS SELECT genre_id, genre_name FROM movie_genres")
print("Tables registered in DuckDB")


## Q1: Content Mix — Movies vs TV Shows

In [ ]:
SELECT 'Movies' AS content_type, COUNT(*) AS total FROM trending_movies
UNION ALL
SELECT 'TV Shows', COUNT(*) FROM popular_tv;


  content_type  total
0       Movies   6131
1     TV Shows   2676

## Q2: Top 10 Genres by Title Count

In [ ]:
SELECT genre_name, COUNT(*) AS title_count
FROM (
    SELECT UNNEST(genre_ids_list) AS gid FROM trending_movies
) t
JOIN genre_lookup gl ON t.gid = gl.genre_id
GROUP BY genre_name
ORDER BY title_count DESC
LIMIT 10;


  genre_name  title_count
0      Drama         5788
1     Comedy          343

![Q2: Top 10 Genres by Title Count](../figures/02_q2_top_genres.png)

## Q3: Average Rating by Genre (min 50 titles)

In [ ]:
SELECT genre_name, ROUND(AVG(vote_average), 2) AS avg_rating, COUNT(*) AS title_count
FROM (
    SELECT vote_average, UNNEST(genre_ids_list) AS gid FROM trending_movies
) t
JOIN genre_lookup gl ON t.gid = gl.genre_id
GROUP BY genre_name
HAVING COUNT(*) > 50
ORDER BY avg_rating DESC;


  genre_name  avg_rating  title_count
0      Drama        6.51         5788
1     Comedy        6.40          343

![Q3: Average Rating by Genre (min 50 titles)](../figures/02_q3_avg_rating_by_genre.png)

## Q4: Hidden Gems — High Rating, Low Popularity

In [ ]:
SELECT title, vote_average, popularity, release_date
FROM trending_movies
WHERE vote_average >= 8.0
  AND popularity < (SELECT quantile_cont(popularity, 0.5) FROM trending_movies)
ORDER BY vote_average DESC, popularity ASC
LIMIT 15;


                                           title  vote_average  popularity release_date
0                                     Seabiscuit          11.6       58.81   2003-06-15
1                                   Project Papa          11.3       82.26   2018-06-15
2                                            VS.          10.9       31.21   2018-06-15
3                David Batra: Elefanten i rummet          10.8       23.17   2020-06-15
4                          Naga The Eternal Yogi          10.8      104.00   2016-06-15
5                                  Airplane Mode          10.7       61.99   2020-06-15
6                                 The One I Love          10.6       10.71   2014-06-15
7                                Holiday on Mars          10.6       28.87   2020-06-15
8                                    Yoga Hosers          10.6       40.40   2016-06-15
9                                 The River Wild          10.6       52.47   1994-06-15
10                              

## Q5: Release Year Trend

In [ ]:
SELECT CAST(SUBSTR(release_date, 1, 4) AS INTEGER) AS release_year,
       COUNT(*) AS movie_count,
       ROUND(AVG(vote_average), 2) AS avg_rating
FROM trending_movies
WHERE release_date IS NOT NULL AND release_date != ''
GROUP BY release_year
HAVING release_year BETWEEN 2000 AND 2021
ORDER BY release_year;


    release_year  movie_count  avg_rating
0           2000           33        6.84
1           2001           40        6.42
2           2002           44        6.36
3           2003           51        6.64
4           2004           55        6.19
5           2005           67        6.24
6           2006           82        6.33
7           2007           74        6.49
8           2008          113        6.41
9           2009          118        6.39
10          2010          154        6.66
11          2011          145        6.63
12          2012          173        6.34
13          2013          225        6.65
14          2014          264        6.41

![Q5: Release Year Trend](../figures/02_q5_release_trend.png)

## Q6: Rating by Vote Count Bucket

In [ ]:
SELECT CASE
    WHEN vote_count < 100 THEN '0-99'
    WHEN vote_count < 1000 THEN '100-999'
    WHEN vote_count < 5000 THEN '1K-4.9K'
    ELSE '5K+'
END AS vote_bucket,
COUNT(*) AS titles,
ROUND(AVG(vote_average), 2) AS avg_rating,
ROUND(AVG(popularity), 2) AS avg_popularity
FROM trending_movies
GROUP BY vote_bucket
ORDER BY avg_rating DESC;


  vote_bucket  titles  avg_rating  avg_popularity
0     100-999     622        6.52          150.23
1     1K-4.9K    3442        6.51          155.67
2         5K+    2047        6.50          151.01
3        0-99      20        6.15          148.95

![Q6: Rating by Vote Count Bucket](../figures/02_q6_vote_bucket_rating.png)

## Q7: TV Show Decade Distribution

In [ ]:
SELECT CASE
    WHEN CAST(SUBSTR(release_date, 1, 4) AS INTEGER) < 1990 THEN '< 1990s'
    WHEN CAST(SUBSTR(release_date, 1, 4) AS INTEGER) < 2000 THEN '1990s'
    WHEN CAST(SUBSTR(release_date, 1, 4) AS INTEGER) < 2010 THEN '2000s'
    WHEN CAST(SUBSTR(release_date, 1, 4) AS INTEGER) < 2020 THEN '2010s'
    ELSE '2020s'
END AS decade,
COUNT(*) AS tv_count,
ROUND(AVG(vote_average), 2) AS avg_rating
FROM popular_tv
WHERE release_date IS NOT NULL AND release_date != ''
GROUP BY decade
ORDER BY decade;


    decade  tv_count  avg_rating
0    1990s        33        6.61
1    2000s       133        6.50
2    2010s      1743        6.47
3    2020s       751        6.53
4  < 1990s        16        6.64

## Q8: Upcoming Releases by Month

In [ ]:
SELECT SUBSTR(release_date, 6, 2) AS release_month,
       COUNT(*) AS upcoming_count,
       ROUND(AVG(vote_average), 2) AS avg_rating
FROM upcoming_movies
WHERE release_date IS NOT NULL AND release_date != ''
GROUP BY release_month
ORDER BY release_month;


  release_month  upcoming_count  avg_rating
0            12             277         6.4

## Q9: Genre Popularity Ranking

In [ ]:
SELECT genre_name, top_movie_popularity, total_movies_in_genre,
       ROUND(top_movie_popularity / NULLIF(total_movies_in_genre, 0), 2) AS pop_per_title
FROM genre_popularity
ORDER BY top_movie_popularity DESC
LIMIT 15;


         genre_name  top_movie_popularity  total_movies_in_genre  pop_per_title
0            Comedy                667.84                    343           1.95
1             Drama                648.04                   5788           0.11
2            Action                  0.00                      0            NaN
3         Adventure                  0.00                      0            NaN
4            Family                  0.00                      0            NaN
5       Documentary                  0.00                      0            NaN
6         Animation                  0.00                      0            NaN
7            Horror                  0.00                      0            NaN
8             Crime                  0.00                      0            NaN
9           Romance                  0.00                      0            NaN
10          Fantasy                  0.00                      0            NaN
11          History                  0.0

## Q10: Content Quality Tiers

In [ ]:
SELECT CASE
    WHEN vote_average >= 8.0 THEN 'Excellent (8.0+)'
    WHEN vote_average >= 7.0 THEN 'Good (7.0–7.9)'
    WHEN vote_average >= 6.0 THEN 'Average (6.0–6.9)'
    ELSE 'Below Average (< 6.0)'
END AS quality_tier,
COUNT(*) AS title_count,
ROUND(AVG(popularity), 2) AS avg_popularity,
ROUND(AVG(vote_count), 0) AS avg_votes
FROM trending_movies
GROUP BY quality_tier
ORDER BY AVG(vote_average) DESC;


            quality_tier  title_count  avg_popularity  avg_votes
0       Excellent (8.0+)          959          151.63     4111.0
1         Good (7.0–7.9)         1345          159.20     4071.0
2      Average (6.0–6.9)         1713          154.77     4113.0
3  Below Average (< 6.0)         2114          149.81     4095.0

![Q10: Content Quality Tiers](../figures/02_q10_quality_tiers.png)

---
**Summary of Key Findings**
- Movies outnumber TV shows 6,131 to 2,676.
- Top genres: Drama, Comedy.
- Highest-rated genre: Drama (6.51 avg).
- 37.6% of movies are rated Good or Excellent.
